# M1 · Road risk: training notebook

**Goal:** from what we know about a road and the moment (+ the weather), predict how risky it is to drive there **right now**, then turn that into 4 warning levels: **low · medium · high · very high**.

**Data:** `training_table_weather.csv` (French BAAC accidents 2018–2024). Each real accident (`label = 1`) is compared with "nothing happened" moments on French roads, picked **like traffic** (many at 17h, few at 3h), about 4 of them per accident. So the model estimates risk **per passing car**, not raw accident counts.

**Steps**
1. Load the training table
2. Choose the inputs (only things known *before* a crash)
3. Split by time: learn from the past, test on the future
4. Train 4 models: a simple rule, a classic formula, our AI model, and our AI model + accident history
5. Compare them
6. Look at what the model learned
7. Turn the score into 4 levels
8. Save the model
9. Try it on example situations

**Kernel:** pick **Python (solvex-m3)** at the top right (same packages as M3). Put the CSV in `m1/data/`. Runs on a normal PC in a few minutes, no GPU needed.

## 1 · Load the data

In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
SEED = 42

# The weather version is preferred; the plain one works too (weather inputs are then skipped)
DATA = next(p for p in [Path("data/training_table_weather.csv"), Path("data/training_table.csv")] if p.exists())
df = pd.read_csv(DATA, parse_dates=["datetime"], low_memory=False)

print(f"Loaded {DATA}: {len(df):,} rows, {df.shape[1]} columns")
print(f"  real accidents: {int(df['label'].sum()):,}   'nothing happened' moments: {int((df['label'] == 0).sum()):,}")
print(f"  share of accidents: {df['label'].mean():.1%}")
print(f"  years: {df['datetime'].dt.year.min()}–{df['datetime'].dt.year.max()}")

## 2 · Choose the inputs

Rule: only things we can know **before** a crash happens, and that we can also get for Algerian roads. Nothing that describes the crash itself (collision type, vehicles involved…).

- **Road** (OpenStreetMap in Algeria): road type, shape, speed limit, intersection, in/out of town, two-way or not, lanes, slope, special infrastructure
- **Time:** hour, weekday, month, weekend, darkness
- **Weather** (Open-Meteo in Algeria): rain now and over 3 h, fog, snow, storm

**History** (accidents at this place in the previous 2 years) is the strongest signal in France, but Algeria has no geolocated accident database. So the main model (*"Algeria mode"*) doesn't use it. We train a second model *with* it only to measure what we lose, and what we gain once our app collects its own crash data.

Road codes are categories (codes, not quantities): `catr = 3` is not "three times" `catr = 1`.

In [ ]:
ROAD = {
    "catr":  "road      type (1 motorway, 2 national, 3 departmental, 4 municipal…)",
    "plan":  "road      shape (1 straight, 2 left curve, 3 right curve, 4 S-bend)",
    "vma":   "road      speed limit (km/h)",
    "int":   "road      intersection (1 none, 2 X, 3 T, 6 roundabout…)",
    "agg":   "road      1 outside town, 2 in town",
    "circ":  "road      1 one-way, 2 two-way, 3 separated carriageways",
    "nbv":   "road      number of lanes",
    "prof":  "road      1 flat, 2 slope, 3 hilltop, 4 bottom of hill",
    "infra": "road      infrastructure (0 none, 1 tunnel, 2 bridge…)",
}
TIME = {
    "hour":       "time      hour of day (0–23)",
    "weekday":    "time      0 Monday … 6 Sunday",
    "month":      "time      1–12",
    "is_weekend": "time      Saturday or Sunday (1/0)",
    "is_dark":    "time      sun is down (1/0)",
    "is_holiday": "time      public holiday (1/0)",
}
WEATHER = {
    "rain_mm":    "weather   rain this hour (mm)",
    "rain_3h_mm": "weather   rain over the last 3 hours (mm)",
    "is_rain":    "weather   raining (1/0)",
    "is_fog":     "weather   fog (1/0)",
    "is_snow":    "weather   snow (1/0)",
    "is_storm":   "weather   storm (1/0)",
}
HISTORY = {
    "hist_n_acc":       "history   accidents here in the previous 2 years",
    "hist_fatal_share": "history   share of them that were fatal",
}
CATEGORICAL = ["catr", "plan", "int", "agg", "circ", "prof", "infra"]
TARGET = "label"

# Keep only the inputs this table actually has
FEATURES = {c: d for c, d in {**ROAD, **TIME, **WEATHER}.items() if c in df.columns}
X_COLS = list(FEATURES)
HIST_COLS = [c for c in HISTORY if c in df.columns]
X_COLS_HIST = X_COLS + HIST_COLS
CATS = [c for c in CATEGORICAL if c in X_COLS]
WITH_WEATHER = any(c in df.columns for c in WEATHER)

for c in CATS:   # set categories on the whole table so train and test share the same codes
    df[c] = df[c].astype("category")

print("Model inputs:")
for col, desc in FEATURES.items():
    print(f"  {col:<11} {desc}")
missing = [c for c in {**ROAD, **TIME, **WEATHER} if c not in df.columns]
if missing:
    print(f"\nNot in this table (skipped): {', '.join(missing)}")
print(f"\nHistory inputs available for the comparison model: {', '.join(HIST_COLS) or 'none'}")
print(f"Target: {TARGET} = 1 if a real accident, 0 if a 'nothing happened' moment")

## 3 · Split by time: learn from the past, test on the future

- **train (2018–2021)**: the model learns from it
- **validation (2022)**: used while training to know when to stop adding trees
- **test (2023–2024)**: never seen until the end; gives the honest score

We split **by date**, not at random: the model will be used on future days, so we check it on years it has never seen. (The Kaggle version stopped training based on the test years, which makes the test score a bit too optimistic. Here the test stays untouched.)

In [ ]:
VALID_FROM, TEST_FROM = "2022-01-01", "2023-01-01"

train = df[df["datetime"] < VALID_FROM]
valid = df[(df["datetime"] >= VALID_FROM) & (df["datetime"] < TEST_FROM)]
test = df[df["datetime"] >= TEST_FROM]

for name, part in [("train", train), ("validation", valid), ("test", test)]:
    years = part["datetime"].dt.year
    print(f"{name:<11} {len(part):>9,} rows  {years.min()}–{years.max()}  accidents: {part[TARGET].mean():.1%}")
assert train["datetime"].max() < test["datetime"].min()
print("\nTest years come strictly after training years ✓")

y_train, y_valid, y_test = train[TARGET], valid[TARGET], test[TARGET]

## 4 · Train the models

| Model | What it is | Why |
|---|---|---|
| **A · Speed-limit rule** | logistic regression on the speed limit only | the simplest possible baseline |
| **B · Classic formula** | logistic regression on all inputs (one weight per road code), learned on a 300k-row sample | what a hand-made scoring grid would do |
| **C · Our AI model** | LightGBM (gradient-boosted decision trees) | learns combinations (a curve is worse at night, 17h is worse in town) and handles missing values |
| **D · AI + history** | same as C + past accidents at the place | only to measure what history is worth |

LightGBM builds hundreds of small decision trees, each one fixing the errors of the previous ones. We stop adding trees when the validation score stops improving.

In [ ]:
def one_hot(frame, cols):
    """Inputs for the logistic regressions: road codes become 0/1 columns, gaps become 0 + a flag."""
    X = pd.get_dummies(frame[cols], columns=[c for c in CATS if c in cols], dummy_na=True, dtype=float)
    num = [c for c in cols if c not in CATS]
    for c in num:
        X[c + "_missing"] = X[c].isna().astype(float)
    X[num] = X[num].fillna(train[num].median())
    return X

print("A · Speed-limit rule ...")
vma_mean, vma_std = train["vma"].mean(), train["vma"].std()
scale_vma = lambda f: ((f[["vma"]].fillna(vma_mean) - vma_mean) / vma_std).to_numpy()
model_a = LogisticRegression().fit(scale_vma(train), y_train)
print(f"   done: weight on the speed limit {model_a.coef_[0][0]:+.2f} (positive = faster roads riskier)")

print("B · Classic formula ...")
b_sample = train.sample(min(len(train), 300_000), random_state=SEED)   # a sample is enough for one weight per input, and keeps memory low
Xb_train = one_hot(b_sample, X_COLS)
b_mean, b_std = Xb_train.mean(), Xb_train.std().replace(0, 1)
prep_b = lambda f: ((one_hot(f, X_COLS).reindex(columns=Xb_train.columns, fill_value=0) - b_mean) / b_std).to_numpy("float32")
model_b = LogisticRegression(max_iter=300).fit(prep_b(b_sample), b_sample[TARGET])
print(f"   done: {Xb_train.shape[1]} weights, learned on {len(b_sample):,} rows")

LGB_PARAMS = dict(
    n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=200,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=SEED, verbose=-1,
)

def train_lgbm(cols, label):
    print(f"{label} ...")
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        train[cols], y_train,
        eval_set=[(valid[cols], y_valid)], eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)],
    )
    print(f"   done: stopped at {model.best_iteration_} trees "
          f"(validation AUC {model.best_score_['valid_0']['auc']:.3f})\n")
    return model

model_c = train_lgbm(X_COLS, "C · Our AI model")
model_d = train_lgbm(X_COLS_HIST, "D · AI + history") if HIST_COLS else None

## 5 · Compare the models on the test years

**AUC**: pick one real accident and one "nothing happened" moment; AUC is the chance the model gives the higher risk to the accident. 0.5 = coin flip, 1.0 = perfect.

**Capture @ 10 % / 20 %**: if we warn drivers only in the 10 % (or 20 %) riskiest situations, what share of real accidents happened in those situations? Random warnings would catch 10 % (or 20 %). This is the number that matters for the app: few warnings, many accidents covered.

In [ ]:
models = {
    "A · Speed-limit rule": lambda f: model_a.predict_proba(scale_vma(f))[:, 1],
    "B · Classic formula":  lambda f: model_b.predict_proba(prep_b(f))[:, 1],
    "C · Our AI model":     lambda f: model_c.predict_proba(f[X_COLS])[:, 1],
}
if model_d is not None:
    models["D · AI + history"] = lambda f: model_d.predict_proba(f[X_COLS_HIST])[:, 1]

def capture_curve(y, p):
    """For each share of riskiest situations warned (x), the share of real accidents caught (y)."""
    y = np.asarray(y)[np.argsort(-p)]
    return np.arange(1, len(y) + 1) / len(y), np.cumsum(y) / y.sum()

def capture_at(y, p, share):
    y = np.asarray(y)[np.argsort(-p)]
    return y[: int(len(y) * share)].sum() / y.sum()

rows, test_probs = [], {}
for name, predict in models.items():
    p = predict(test)
    test_probs[name] = p
    rows.append({
        "model": name,
        "AUC": roc_auc_score(y_test, p),
        "capture @ 10%": capture_at(y_test, p, 0.10),
        "capture @ 20%": capture_at(y_test, p, 0.20),
    })
results = pd.DataFrame(rows).set_index("model")
print(results.round(3).to_string())

c = results.loc["C · Our AI model"]
print(f"\nWarning in the 10% riskiest situations covers {c['capture @ 10%']:.0%} of real accidents "
      f"({c['capture @ 10%'] / 0.10:.1f}x better than random).")
if model_d is not None:
    d = results.loc["D · AI + history"]
    print(f"Without accident history we lose {d['AUC'] - c['AUC']:.3f} AUC: "
          "this is what our own crash data will bring back over time.")

fig, ax = plt.subplots(figsize=(6, 5))
for name, p in test_probs.items():
    x, y = capture_curve(y_test, p)
    step = max(1, len(x) // 2000)
    ax.plot(x[::step] * 100, y[::step] * 100, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
ax.plot([0, 100], [0, 100], "k--", lw=0.8, label="random warnings")
ax.set_xlabel("% of situations warned (riskiest first)")
ax.set_ylabel("% of real accidents covered")
ax.set_title("Capture curve on 2023–2024")
ax.legend(fontsize=8)
plt.show()

## 6 · What the AI model learned

**Importance**: how much each input helped the trees separate accidents from normal moments.

**Risk curves**: we invent situations and ask the model, hour by hour. This shows the model behaves sensibly (riskier at night per passing car, rural curves worse than straight town streets).

In [ ]:
imp = pd.Series(model_c.booster_.feature_importance("gain"), index=X_COLS)
imp = (imp / imp.sum()).sort_values()
print("Share of the model's decisions explained by each input:")
print((imp[::-1] * 100).round(1).astype(str).add(" %").to_string())

BASE_RATE = float(y_train.mean())   # ≈ 1 accident for 4 normal moments, by construction of the table

def x_vs_average(p):
    """How much riskier than an average situation (1.0 = average, 2.0 = twice, 0.5 = half)."""
    p = np.asarray(p)
    return (p / (1 - p)) / (BASE_RATE / (1 - BASE_RATE))

def situation(**road_and_time):
    """One row per value: unknown inputs stay empty (the model handles gaps)."""
    n = max(len(np.atleast_1d(v)) for v in road_and_time.values())
    x = pd.DataFrame({c: np.full(n, np.nan) for c in X_COLS})
    for k, v in road_and_time.items():
        if k in x:
            x[k] = np.broadcast_to(v, n).astype(float)
    if "is_weekend" in x and "weekday" in road_and_time:
        x["is_weekend"] = (x["weekday"] >= 5).astype(float)
    for c in CATS:
        x[c] = pd.Categorical(x[c], categories=df[c].cat.categories)
    return x[X_COLS]

hours = np.arange(24)
dark = ((hours < 7) | (hours >= 20)).astype(float)
town = dict(catr=4, vma=50, nbv=2, circ=2, agg=2, plan=1, int=1, prof=1, infra=0)
rural_curve = dict(catr=3, vma=80, nbv=2, circ=2, agg=1, plan=2, int=1, prof=2, infra=0)
dry = dict(rain_mm=0, rain_3h_mm=0, is_rain=0, is_fog=0, is_snow=0, is_storm=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
imp.plot.barh(ax=axes[0], color="tab:blue")
axes[0].set_title("Input importance (model C)")
for label, road, wkday in [("town street, Tuesday", town, 1), ("town street, Saturday", town, 5),
                           ("rural curve 80 km/h, Tuesday", rural_curve, 1),
                           ("rural curve 80 km/h, Saturday", rural_curve, 5)]:
    x = situation(**road, **dry, hour=hours, weekday=wkday, month=10, is_dark=dark)
    axes[1].plot(hours, x_vs_average(model_c.predict_proba(x)[:, 1]), marker="o", ms=3, label=label)
axes[1].axhline(1, color="k", lw=0.8, ls="--")
axes[1].set_xlabel("hour of day")
axes[1].set_ylabel("risk per passing car (x vs average)")
axes[1].set_title("Predicted risk by hour (October, dry)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7 · From a score to 4 levels

The table has about 1 accident for 4 normal moments, so the raw probability is not a real-world chance. We turn it into **"x vs average"**: how many times riskier than an average situation in the data. Then:

| Level | Rule | What the app does |
|---|---|---|
| **very high** | ≥ 3x average | voice warning before entering the road |
| **high** | 1.5x – 3x | visible warning |
| **medium** | 0.7x – 1.5x | shown on the map only |
| **low** | < 0.7x | nothing |

Then we check on the test years: how many situations land in each level, and what share of real accidents they hold.

In [ ]:
LEVEL_CUTS = {"medium": 0.7, "high": 1.5, "very high": 3.0}
LEVELS = ["low", "medium", "high", "very high"]

def to_level(p):
    x = x_vs_average(p)
    return np.select([x >= LEVEL_CUTS["very high"], x >= LEVEL_CUTS["high"], x >= LEVEL_CUTS["medium"]],
                     ["very high", "high", "medium"], default="low")

lvl = pd.Categorical(to_level(test_probs["C · Our AI model"]), LEVELS)
summary = pd.DataFrame({
    "% of situations": pd.Series(lvl).value_counts(normalize=True).reindex(LEVELS) * 100,
    "% of real accidents": pd.Series(lvl[y_test.values == 1]).value_counts(normalize=True).reindex(LEVELS) * 100,
})
summary["accident rate %"] = pd.Series(y_test.values).groupby(np.asarray(lvl)).mean().reindex(LEVELS) * 100
print("Test years 2023–2024, model C:")
print(summary.round(1).to_string())
warned = np.isin(lvl, ["high", "very high"])
print(f"\nWarnings (high + very high) fire in {warned.mean():.0%} of situations "
      f"and cover {warned[y_test.values == 1].mean():.0%} of real accidents.")

## 8 · Save the model

We save model C plus everything needed to use it: the input list, the categories, the level cuts and the test scores. Same file names and JSON keys as the Kaggle version, so the backend doesn't change.

In [ ]:
out = Path("models")
out.mkdir(exist_ok=True)
NAME = "risk_model_nohist"
model_c.booster_.save_model(out / f"{NAME}.txt", num_iteration=model_c.best_iteration_)
config = {
    "features": X_COLS,
    "feature_notes": FEATURES,
    "categorical": CATS,
    "with_weather": WITH_WEATHER,
    "with_history": False,
    "base_rate": round(BASE_RATE, 4),
    "level_cuts_x_vs_average": LEVEL_CUTS,
    "levels": LEVELS,
    "target": "accident vs traffic-like 'nothing happened' moment, French BAAC; train 2018-21, valid 2022, test 2023-24",
    "test_auc": round(float(c["AUC"]), 4),
    "top10_capture": round(float(c["capture @ 10%"]), 4),
    "top20_capture": round(float(c["capture @ 20%"]), 4),
    "trees": int(model_c.best_iteration_),
}
(out / f"{NAME}.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
print("Saved:")
for f in sorted(out.iterdir()):
    print(f"  {f}  ({f.stat().st_size / 1024:.0f} KB)")

## 9 · Try it on example situations

This is what the backend will do for the app: load the saved files (not the notebook's model), describe the road and the moment, ask the model, print the level. Codes are the same as the French data (see section 2).

In [ ]:
booster = lgb.Booster(model_file=str(out / f"{NAME}.txt"))
saved = json.loads((out / f"{NAME}.json").read_text())
assert saved["features"] == X_COLS

EXAMPLES = {
    "City street, Friday 17h (rush hour)":
        dict(**town, hour=17, weekday=4, month=10, is_dark=0),
    "Same city street, Tuesday 4h (empty, dark)":
        dict(**town, hour=4, weekday=1, month=10, is_dark=1),
    "City roundabout, Friday 17h":
        dict(**{**town, "int": 6}, hour=17, weekday=4, month=10, is_dark=0),
    "Rural curve, 80 km/h, Saturday 2h (dark)":
        dict(**rural_curve, hour=2, weekday=5, month=7, is_dark=1),
    "Same rural curve, Tuesday 11h (daylight)":
        dict(**rural_curve, hour=11, weekday=1, month=7, is_dark=0),
    "Same rural curve, Tuesday 11h, heavy rain":
        dict(**rural_curve, hour=11, weekday=1, month=11, is_dark=0,
             rain_mm=6, rain_3h_mm=15, is_rain=1),
    "National road (like an Algerian RN), 90 km/h, Sunday 23h":
        dict(catr=2, vma=90, nbv=2, circ=2, agg=1, plan=1, int=3, prof=1, infra=0,
             hour=23, weekday=6, month=8, is_dark=1),
    "Motorway, 130 km/h, 3 lanes, Wednesday 10h":
        dict(catr=1, vma=130, nbv=3, circ=3, agg=1, plan=1, int=1, prof=1, infra=0,
             hour=10, weekday=2, month=5, is_dark=0),
}

X = pd.concat([situation(**{**dry, **s}) for s in EXAMPLES.values()], ignore_index=True)
p = booster.predict(X)
res = pd.DataFrame({"situation": list(EXAMPLES), "x vs average": x_vs_average(p).round(2), "level": to_level(p)})
print(res.sort_values("x vs average", ascending=False).to_string(index=False))

## Notes for the pitch

- Trained on **~1.8 M rows** of French accidents vs normal traffic moments, tested on **2 years it never saw** (2023–2024).
- No accident history needed (*Algeria mode*): only road data from OpenStreetMap, time, and weather from Open-Meteo. History adds accuracy (model D), and our app's own crash reports bring it back over time.
- The table is built with ~4 normal moments per accident, so read the score as **"x times riskier than average"**, not as a real-world chance. The levels are what the app uses.
- Limits to say honestly: French roads and drivers, traffic is approximated by a typical daily curve (no real traffic counts), weather inputs add little once the hour and the road are known.